# Brazil adaptation HIAP ranking

Ranks the 38 adaptation actions in the ICB × OEF action bank for each Brazilian city, then compares the result with the earlier HIAP model.

**overall = impact × 55% + feasibility × 22.5% + alignment × 22.5%**, where **feasibility = cost × 50% + legal × 50%**. Every pillar is on a 0–1 scale; only impact varies by city.

| Part | What it does | Writes |
| --- | --- | --- |
| 1. Inputs | Cleans each source into one file per input, keyed on `action_id` | `data/input/` |
| 2. City impact | Scores each action against the city's AdaptaBrasil risk | `data/output/city_ranking.csv` |
| 3. Overall ranking | Combines the three pillars and writes the workbook | `data/output/overall_ranking.csv`, `br_adaptation_hiap_ranking.xlsx` |
| 4. Earlier model | Compares each city's top 20 with the earlier model's | `data/output/comparison_earlier_model.csv` |

Run from this folder. Needs `pandas` and `openpyxl`, the source files in `sample/` (gitignored), the policy alignment v2 results, and access to the CityCatalyst global API for any city not yet in `data/input/adaptabrasil_risk.csv`.

In [1]:
from pathlib import Path
import json, unicodedata, urllib.parse, urllib.request, warnings
import openpyxl
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

HERE = Path.cwd()
SAMPLE = HERE / "sample"
INPUT, MAPPING, OUTPUT = (HERE / "data" / d for d in ("input", "mapping", "output"))
INPUT.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)

V15 = SAMPLE / "Adaptation_Action_Impact_Assessment_ICB_x_OEF_V15.xlsx"
COST = SAMPLE / "Action_Cost_Calculation.xlsx"
LEGAL = SAMPLE / "legal-feasibility-score-profiles_2.xlsx"
OLD_MODEL = SAMPLE / "hiap_actions_extract_2026-10-12.csv"
ALIGNMENT = HERE.parents[3] / "br-federal-government/br-climate-policy-documents/releases/v2/data/results.csv"
API = "https://ccglobal.openearth.dev"
WORKBOOK = HERE / "br_adaptation_hiap_ranking.xlsx"

# Cities to rank, as "name": "state". The first five are the I Care workbook's test cities.
CITIES = {"Caxias do Sul": "RS", "Sobral": "CE", "Santa Cruz do Arari": "PA", "São Luís": "MA", "Itaubal": "AP"}
TEST_CITIES = ["Caxias do Sul", "Sobral", "Santa Cruz do Arari", "São Luís", "Itaubal"]

# Defaults. All of these can also be changed on the workbook's Settings tab.
WEIGHTS = {"impact": 0.55, "feasibility": 0.225, "alignment": 0.225}
FEAS_SPLIT = {"cost": 0.5, "legal": 0.5}
COST_SCORE = {"low": 0.6, "medium": 0.5, "high": 0.4}   # methodology scale (for review in pilot testing); medium = missing = 0.5
RANK_ONLY_WITH_IMPACT = True   # actions with no impact score get no rank, as in the I Care workbook

v15 = openpyxl.load_workbook(V15, data_only=True)

## 1. Inputs

### 1.1 Actions

The 38 updated actions, from the V15 workbook's *Final Action Bank*. Every other input is checked against this list.

In [2]:
bank = pd.DataFrame(
    [r[:5] for r in v15["Final Action Bank"].iter_rows(min_row=3, values_only=True) if r[0]],
    columns=["action_id", "action_name", "description", "timeline", "cost_band"],
)
assert len(bank) == 38 and bank.action_id.is_unique
ACTIONS = set(bank.action_id)

actions = bank.drop(columns="cost_band")
names = actions.set_index("action_id").action_name
actions.to_csv(INPUT / "actions.csv", index=False)

### 1.2 Impact assessment

One row per action × sector × component (vulnerability or exposure): 38 × 6 × 2 = 456 rows, from the tab *Action x sectors assessments*. Only **Direct** links get a component score:

`effectiveness^0.5 × confidence^0.25 × timeline^0.2 × maladaptation^0.05`

The tab's row-3 labels sit one column to the left of their data, so columns are read by letter. `human_reviewed` is true where any reviewer column is filled.

**Check:** 12 rows per action, one timeline per action matching the action list, scores only on Direct links, and every score matches the formula.

In [3]:
COLS = {"action_id": "B", "sector": "F", "component": "H", "eligibility": "L", "eligibility_rationale": "M",
        "effectiveness": "S", "effectiveness_rationale": "U", "confidence": "AA", "confidence_rationale": "AC",
        "maladaptation": "AI", "maladaptation_rationale": "AK", "timeline": "AQ", "component_score": "AT"}
REVIEW_COLS = ["N", "Q", "V", "Y", "AD", "AL"]

ws = v15["Action x sectors assessments"]
impact = pd.DataFrame([
    {**{k: ws[f"{c}{r}"].value for k, c in COLS.items()},
     "human_reviewed": any(ws[f"{c}{r}"].value not in (None, "") for c in REVIEW_COLS)}
    for r in range(4, ws.max_row + 1) if ws[f"B{r}"].value
])
for c in ["effectiveness", "confidence"]:
    impact[c] = impact[c].str.replace(f" {c}", "", regex=False)
impact["maladaptation"] = impact.maladaptation.map({"Maladaptation signal": "Signal", "No maladaptation signal": "None"})
impact["component_score"] = impact.component_score.astype(float)
impact = impact[["action_id", "sector", "component", "eligibility", "effectiveness", "confidence", "maladaptation",
                 "timeline", "component_score", "human_reviewed", "eligibility_rationale",
                 "effectiveness_rationale", "confidence_rationale", "maladaptation_rationale"]]

assert set(impact.action_id) == ACTIONS and len(impact) == 456
assert impact.groupby("action_id").timeline.first().equals(bank.set_index("action_id").timeline.sort_index())
direct = impact[impact.eligibility == "Direct"]
assert direct.component_score.notna().all() and impact[impact.eligibility != "Direct"].component_score.isna().all()
calc = (direct.effectiveness.map({"High": 1, "Medium": .5, "Low": .25, "No demonstrated effectiveness": 0}) ** .5
        * direct.confidence.map({"High": 1, "Medium": .75, "Low": .5}) ** .25
        * direct.timeline.map({"<5 years": 1, "5-10 years": .75, ">10 years": .5}) ** .2
        * direct.maladaptation.map({"None": 1, "Signal": .75}) ** .05)
assert (calc.round(4) - direct.component_score).abs().max() < 1e-3

impact.to_csv(INPUT / "impact.csv", index=False)
print(f"{len(direct)} scored links, {impact.human_reviewed.sum()} rows human-reviewed")
print(f"no Direct link, so cannot be ranked: {sorted(ACTIONS - set(direct.action_id))}")

54 scored links, 44 rows human-reviewed
no Direct link, so cannot be ranked: ['icare_0164', 'icare_0165', 'icare_0171', 'icare_0172', 'icare_0182', 'icare_0185', 'ipcc_0088', 'ipcc_0089', 'ipcc_0104']


### 1.3 Cost

From the *Action Cost Calculation* workbook. Four components, CTI, IR, E and T (not defined in the workbook), give a cost index `IC = round((CTI + IR) ÷ 2 + E + T)` from 3 to 8: 3–4 low, 5–6 medium, 7–8 high. Cost is the only finance field used to rank (agreed with partners, September 2026).

**Check:** the index and band follow from the components, and the bands match the V15 action list.

In [4]:
cost = pd.DataFrame(
    [r[:9] for r in openpyxl.load_workbook(COST, data_only=True).worksheets[0].iter_rows(min_row=5, values_only=True) if r[0]],
    columns=["action_id", "action_name", "description", "cti", "ir", "e", "t", "ic", "cost_band"],
).drop(columns=["action_name", "description"])
cost["cost_band"] = cost.cost_band.str.lower()

assert set(cost.action_id) == ACTIONS and cost.action_id.is_unique
assert (((cost.cti + cost.ir) / 2 + cost.e + cost.t + 0.5).astype(int) == cost.ic).all()   # Excel ROUND: halves up
assert (pd.cut(cost.ic, [0, 4, 6, 8], labels=["low", "medium", "high"]).astype(str) == cost.cost_band).all()
assert (cost.cost_band.values == cost.action_id.map(bank.set_index("action_id").cost_band.str.lower()).values).all()

cost.to_csv(INPUT / "actions_cost.csv", index=False)
cost.cost_band.value_counts()

cost_band
medium    26
high       8
low        4
Name: count, dtype: int64

### 1.4 Legal feasibility

From the *Legal feasibility score profiles* workbook: a score from 1 to 5 for how much authority a municipality has to carry out the action, and its competence to make rules about it, under Brazilian federal law. Each action lists the federal laws that apply, and each law is graded for

- *authority*: total (1) or partial (0.5)
- *competence*: exclusive (1), supplementary (0.66) or concurrent/private (0.33)

Each grade is averaged over the action's laws and stretched to 1–5; the score is the mean of the two. Grades: 5 Highly feasible, from 4.2 Feasible, from 3.3 Feasible with adjustments, from 2.5 Feasible with heavy adjustments, above 1.5 Least feasible, otherwise Not feasible.

**Check.** The score is recalculated two ways. *As delivered* copies the workbook's formula and must match its scores. *As described* follows the method above. They differ because the workbook formula

1. looks for partial authority as `parcial`, but the laws are tagged `partial`, so it counts as 0 instead of 0.5;
2. subtracts 0.33 from the competence total, not the average, so an action whose laws are all concurrent/private gets about 2.2 instead of 1.

The ranking uses the score as delivered until the author confirms. The table shows the scores a corrected formula would change.

In [5]:
lwb = openpyxl.load_workbook(LEGAL, data_only=True)
legal = pd.DataFrame(
    [r[:5] for r in lwb["Summary"].iter_rows(min_row=5, values_only=True) if r[0]],
    columns=["action_id", "action_name", "description", "legal_score", "legal_grade"],
)[["action_id", "legal_score", "legal_grade"]]
assert set(legal.action_id) == ACTIONS and legal.action_id.is_unique

law_cols = ["action_id", "norm_id", "norm_name", "norm_code", "norm_type", "authority", "competence", "validation"]
laws = pd.concat([pd.DataFrame([r[:8] for r in lwb[a].iter_rows(min_row=15, values_only=True) if r[1]], columns=law_cols)
                  for a in legal.action_id], ignore_index=True)
norms = pd.DataFrame(lwb["norm_database"].iter_rows(min_row=6, values_only=True),
                     columns=["norm_id", "norm_name", "norm_code", "norm_description", "norm_type",
                              "validation", "validation_date", "norm_link"]).dropna(subset=["norm_id"])
laws = laws.merge(norms[["norm_id", "norm_description", "validation_date", "norm_link"]], on="norm_id", how="left")
laws = laws[["action_id", "norm_id", "norm_name", "norm_code", "norm_type", "norm_description", "authority",
             "competence", "validation", "validation_date", "norm_link"]]
assert set(laws.action_id) == ACTIONS
assert laws.authority.isin(["total", "partial"]).all()
assert laws.competence.isin(["exclusive", "supplementary", "concurrent/private"]).all()

COMPETENCE = {"exclusive": 1, "supplementary": 0.66, "concurrent/private": 0.33}

def legal_score(d, as_delivered):
    n = len(d)
    auth = d.authority.map({"total": 1, "partial": 0 if as_delivered else 0.5}).mean()
    comp = d.competence.map(COMPETENCE).sum()
    comp = (comp - 0.33) / (n - 0.33) if as_delivered else (comp / n - 0.33) / (1 - 0.33)
    return int((auth * 4 + 1 + comp * 4 + 1) / 2 * 10 + 0.5 + 1e-9) / 10   # Excel ROUND to 1 decimal

def grade(s):
    return ("Highly feasible" if s == 5 else "Feasible" if s >= 4.2 else "Feasible with adjustments" if s >= 3.3
            else "Feasible with heavy adjustments" if s >= 2.5 else "Least feasible" if s > 1.5 else "Not feasible")

check = legal.set_index("action_id")
by_action = laws.groupby("action_id")[["authority", "competence"]]
check["as_delivered"] = by_action.apply(legal_score, True)
check["as_described"] = by_action.apply(legal_score, False)
assert (check.as_delivered == check.legal_score).all() and (check.legal_score.map(grade) == check.legal_grade).all()

legal.to_csv(INPUT / "legal_assessment_summary.csv", index=False)
laws.to_csv(INPUT / "legal_assessment_complete.csv", index=False)

changed = check[check.as_described != check.legal_score]
print(f"{len(laws)} action × law rows, {laws.norm_id.nunique()} laws. A corrected formula changes {len(changed)} of 38 scores "
      f"(rank correlation {check.legal_score.rank().corr(check.as_described.rank()):.2f}).")
changed.assign(action=changed.index.map(names), change=changed.as_described - changed.legal_score)[
    ["action", "legal_score", "as_described", "change"]].sort_values("change")

230 action × law rows, 52 laws. A corrected formula changes 33 of 38 scores (rank correlation 0.92).


,action,legal_score,as_described,change
action_id,,,,
icare_0184,"Stabilize, inspect and maintain slopes in land...",4.2,3.8,-0.4
ipcc_0103,"Strengthen cooperative, intersectoral and part...",4.2,3.8,-0.4
ipcc_0094,"Strengthen healthcare capacity, continuity and...",4.2,3.8,-0.4
new0002,Strengthen selective waste collection to reduc...,4.2,3.8,-0.4
ipcc_0078,Implement urban cooling and shading measures i...,4.3,4.0,-0.3
icare_0185,Strengthen surveillance and early detection of...,4.3,4.0,-0.3
icare_0180,Expand urban green-blue networks and corridors,4.3,4.0,-0.3
icare_0172,Develop and implement an integrated climate ac...,4.3,4.0,-0.3
ipcc_0098,Strengthen sustainable food value chains,4.3,4.0,-0.3


### 1.5 Policy alignment

From policy alignment v2: the grade, the 0–1 score, the documents behind it and the deciding passage. These are drafts; no rows have been reviewed yet.

In [6]:
alignment = pd.read_csv(ALIGNMENT)[[
    "action_id", "action_name", "grade", "score", "document_support", "documents_medium_or_better",
    "best_document", "best_pdf_page", "decisive_text", "explanation", "rows_reviewed"]]
assert set(alignment.action_id) == ACTIONS and alignment.action_id.is_unique

alignment.to_csv(INPUT / "policy_alignment.csv", index=False)
alignment.grade.value_counts()

grade
Strong    26
Medium    10
Weak       2
Name: count, dtype: int64

## 2. City impact

Follows steps 5–10 of the V15 workbook's *Formulas* tab, with city risk from the CityCatalyst global API instead of numbers pasted into the workbook. That way any Brazilian municipality can be ranked.

| Step | What | Formula |
| --- | --- | --- |
| 5 | Raw action impact per sector | `1 − (1 − SV) × (1 − SE)` |
| 7 | Fit: does the action target the city's weak component? | `(V × SV + E × SE) ÷ (SV + SE)` |
| 8 | Local relevance | `overall risk × fit` |
| 9 | Risk-weighted potential | `raw impact × relevance` |
| 10 | Keep each action's best risk, divide by the city's top score, rank (equal scores share a rank) | |

SV and SE are the action's vulnerability and exposure scores for the sector (Direct links only). Overall risk, V and E are the city's AdaptaBrasil values for the risk.

In [7]:
sector_scores = (direct.pivot_table(index=["action_id", "sector"], columns="component", values="component_score")
                 .fillna(0).rename(columns={"Vulnerability": "sv", "Exposure": "se"}).reset_index())
sector_scores["raw_impact"] = 1 - (1 - sector_scores.sv) * (1 - sector_scores.se)
print(f"{len(sector_scores)} action × sector pairs, {sector_scores.action_id.nunique()} actions")

47 action × sector pairs, 29 actions


### City risk from the API

For each city: find its locode with `/api/v1/cities/search`, then fetch its AdaptaBrasil risk from `/api/v1/cities/{locode}/climate-risk/adapta`. The result is one row per city, sector and risk with the overall risk, hazard (context only), exposure and vulnerability. If a risk has several time frames, the latest is kept. Energy security *Availability* has no exposure or vulnerability, so it drops out, as in the workbook.

Responses are saved to `data/input/adaptabrasil_risk.csv`, and only cities not already in it are fetched. Delete the file to fetch everything again.

In [8]:
def get(path, **params):
    url = API + urllib.parse.quote(path) + ("?" + urllib.parse.urlencode(params) if params else "")
    with urllib.request.urlopen(url, timeout=60) as resp:
        return json.load(resp)

def plain(s):
    return unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode().lower().strip()

def city_risk(city, state):
    hits = [h for h in get("/api/v1/cities/search", q=city, country_code="BR", limit=20)["data"]
            if plain(h["city_name"]) == plain(city) and h["region_code"] == state]
    assert len(hits) == 1, f"{city} ({state}): {len(hits)} matches"
    locode = hits[0]["locode"]
    items = pd.DataFrame(get(f"/api/v1/cities/{locode}/climate-risk/adapta", level="summary")["data"])
    items = items[items.timeframe == items.groupby("risk_id").timeframe.transform("max")]
    keys = ["timeframe", "sector_name", "risk_name"]
    component = items.risk_component_name.map({"Hazard": "hazard", "Climate threat": "hazard",
                                               "Exposure": "exposure", "Vulnerability": "vulnerability"})
    wide = items.assign(component=component).dropna(subset=["component"]).pivot_table(
        index=keys, columns="component", values="risk_component_value_numeric", aggfunc="first")
    overall = items.groupby(keys).risk_value_numeric.first().rename("overall")
    return (wide.join(overall, how="right").reset_index()
            .rename(columns={"sector_name": "sector", "risk_name": "risk"})
            .assign(city=city, state=state, locode=locode))

RISK_FILE = INPUT / "adaptabrasil_risk.csv"
RISK_COLS = ["city", "state", "locode", "timeframe", "sector", "risk", "overall", "hazard", "exposure", "vulnerability"]
saved = pd.read_csv(RISK_FILE) if RISK_FILE.exists() else pd.DataFrame(columns=RISK_COLS)
missing = [c for c in CITIES if c not in set(saved.city)]
if missing:
    fetched = pd.concat([city_risk(c, CITIES[c]) for c in missing], ignore_index=True)
    saved = pd.concat([saved, fetched[RISK_COLS]], ignore_index=True)
    saved.to_csv(RISK_FILE, index=False)

risk = saved[saved.city.isin(CITIES)]
values = risk[["overall", "exposure", "vulnerability"]]
assert (values.isna() | values.ge(0) & values.le(1)).all().all()
risk.groupby("city").size().rename("risks")

city
Caxias do Sul          12
Itaubal                12
Santa Cruz do Arari    12
Sobral                 12
São Luís               12
Name: risks, dtype: int64

### Ranking (steps 7–10)

**Check:** for the five test cities, every rank and normalised score must match the V15 workbook's *Municipal Impact Rank* and *Normalized Impact Pillar Score*. A match means the API serves the same AdaptaBrasil values the workbook used.

In [9]:
m = (risk.dropna(subset=["overall", "exposure", "vulnerability"]).assign(key=lambda d: d.sector.str.lower())
     .merge(sector_scores.assign(key=sector_scores.sector.str.lower()).drop(columns="sector"), on="key"))
m["fit"] = (m.vulnerability * m.sv + m.exposure * m.se) / (m.sv + m.se)
m["potential"] = m.raw_impact * m.overall * m.fit

city_rank = m.loc[m.groupby(["city", "action_id"]).potential.idxmax(),
                  ["city", "locode", "action_id", "sector", "risk", "potential"]].reset_index(drop=True)
city_rank["score"] = city_rank.potential / city_rank.groupby("city").potential.transform("max")
city_rank["rank"] = city_rank[city_rank.potential > 0].groupby("city").score.rank(method="dense", ascending=False)
city_rank.insert(3, "action_name", city_rank.action_id.map(names))

# Check against the workbook
ws = v15["Impact score by city "]
wbk = pd.DataFrame([(r[1], r[4], r[25], r[26]) for r in ws.iter_rows(min_row=3, values_only=True) if r[1]],
                   columns=["city", "action_id", "score_wb", "rank_wb"])
wbk[["score_wb", "rank_wb"]] = wbk[["score_wb", "rank_wb"]].apply(pd.to_numeric, errors="coerce")
wbk = wbk.groupby(["city", "action_id"], as_index=False).agg(score_wb=("score_wb", "max"), rank_wb=("rank_wb", "min"))
cmp = city_rank[city_rank.city.isin(TEST_CITIES)].merge(wbk, on=["city", "action_id"], how="outer")
assert (cmp["rank"].fillna(-1) == cmp.rank_wb.fillna(-1)).all(), "ranks differ from the workbook"
assert (cmp.score - cmp.score_wb).abs().max() < 1e-9, "scores differ from the workbook"
print(f"{len(cmp)} test-city results match the workbook")

city_rank = city_rank.sort_values(["city", "rank"])
city_rank[["city", "locode", "action_id", "action_name", "rank", "score", "sector", "risk", "potential"]].to_csv(
    OUTPUT / "city_ranking.csv", index=False)

190 test-city results match the workbook


## 3. Overall ranking

One row per city and action. Impact is the city's normalised impact score (0 where the action has no Direct link). Feasibility combines the cost band and the legal score rescaled from 1–5 to 0–1. Alignment is the policy alignment score. Equal scores share a rank (1, 1, 2).

In [10]:
feasibility = actions[["action_id", "action_name"]].merge(cost[["action_id", "cost_band"]]).merge(legal)
feasibility["score"] = (FEAS_SPLIT["cost"] * feasibility.cost_band.map(COST_SCORE)
                        + FEAS_SPLIT["legal"] * (feasibility.legal_score - 1) / 4)

city_order = list(CITIES)
grid = pd.MultiIndex.from_product([city_order, actions.action_id], names=["city", "action_id"]).to_frame(index=False)
grid = grid.merge(city_rank[["city", "action_id", "score", "rank", "sector", "risk"]], how="left")
grid["action_name"] = grid.action_id.map(names)
grid["locode"] = grid.city.map(city_rank.drop_duplicates("city").set_index("city").locode)
grid["impact"] = grid.score.fillna(0.0)
grid["feasibility"] = grid.action_id.map(feasibility.set_index("action_id").score)
grid["alignment"] = grid.action_id.map(alignment.set_index("action_id").score)
grid["overall"] = sum(w * grid[k] for k, w in WEIGHTS.items())
ranked = grid[grid.impact > 0] if RANK_ONLY_WITH_IMPACT else grid
grid["overall_rank"] = ranked.overall.round(9).groupby(ranked.city).rank(method="dense", ascending=False)  # round: float noise must not break ties

grid.rename(columns={"overall_rank": "rank", "rank": "impact_rank"})[
    ["city", "locode", "action_id", "action_name", "impact", "feasibility", "alignment", "overall", "rank"]
].to_csv(OUTPUT / "overall_ranking.csv", index=False)

order = grid.groupby("action_id").overall_rank.mean().sort_values(na_position="last").index.tolist()
grid.pivot(index="action_id", columns="city", values="overall_rank")[city_order].loc[order].head(10)

city,Caxias do Sul,Sobral,Santa Cruz do Arari,São Luís,Itaubal
action_id,,,,,
ipcc_0101,1.0,3.0,3.0,1.0,16.0
ipcc_0094,3.0,11.0,5.0,3.0,4.0
ipcc_0099,9.0,6.0,8.0,6.0,6.0
icare_0181,4.0,2.0,12.0,5.0,14.0
new0002,2.0,12.0,4.0,2.0,19.0
icare_0169,7.0,15.0,2.0,16.0,2.0
new0001,5.0,7.0,14.0,9.0,8.0
icare_0183,6.0,21.0,1.0,17.0,1.0
icare_0162,8.0,4.0,17.0,4.0,15.0


### The workbook

`br_adaptation_hiap_ranking.xlsx` does the same calculation in formulas, so changing the weights, the cost/legal split, the cost-band scores or the impact rule on its *Settings* tab re-ranks everything. Python writes the inputs and sets the row order of the first tab (mean rank at the default settings).

| Tab | What it shows |
| --- | --- |
| Overall ranking | Each action's rank in each city, best and mean rank |
| City scores | Per city and action: the three pillar scores, overall score and rank (formulas) |
| Impact | City impact score and rank, and the risk behind it |
| Impact assessment | The action × sector links behind the impact score (Direct and Indirect) |
| Feasibility | Cost band, legal score and grade, and the feasibility score |
| Legal basis | The federal laws behind each legal score |
| Alignment | Policy alignment grade, score and deciding passage |
| Settings | Weights, cost/legal split, cost-band scores, impact rule |

Excel recalculates on opening. At the default settings its ranks equal the table above.

In [11]:
from openpyxl import Workbook
from openpyxl.formatting.rule import ColorScaleRule
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter as col
from openpyxl.worksheet.datavalidation import DataValidation

FONT = "Arial"
PLAIN, BOLD = Font(name=FONT), Font(name=FONT, bold=True)
BLUE, GREEN = Font(name=FONT, color="0000FF"), Font(name=FONT, color="008000")   # input / lookup
GREY = Font(name=FONT, italic=True, color="595959")
HEAD = PatternFill("solid", fgColor="DCE6F0")
YELLOW = PatternFill("solid", fgColor="FFF2CC")

wb = Workbook()
wb.remove(wb.active)

def sheet(name, title, note, columns):
    """New tab: title, note, and a header row (row 3) from [(header, width), ...]."""
    ws = wb.create_sheet(name)
    ws["A1"], ws["A2"] = title, note
    ws["A1"].font, ws["A2"].font = Font(name=FONT, bold=True, size=13), GREY
    for i, (h, w) in enumerate(columns, start=1):
        c = ws.cell(row=3, column=i, value=h)
        c.font, c.fill, c.alignment = BOLD, HEAD, Alignment(wrap_text=True, vertical="bottom")
        ws.column_dimensions[col(i)].width = w
    ws.freeze_panes = "A4"
    return ws

def put(ws, df, wrap=(), fmt=None):
    """Write df from row 4; returns the last row."""
    for r, row in enumerate(df.itertuples(index=False), start=4):
        for c, v in enumerate(row, start=1):
            cell = ws.cell(row=r, column=c, value=None if pd.isna(v) else v)
            cell.font = PLAIN
            if c in wrap:
                cell.alignment = Alignment(wrap_text=True, vertical="top")
            if fmt and c in fmt:
                cell.number_format = fmt[c]
    last = len(df) + 3
    ws.auto_filter.ref = f"A3:{col(len(df.columns))}{last}"
    return last

by_city = lambda df, *keys: df.assign(_c=pd.Categorical(df.city, city_order, ordered=True)).sort_values(
    ["_c", *keys], na_position="last").drop(columns="_c")

# Tabs in display order
ws_over = wb.create_sheet("Overall ranking")
ws_city = sheet("City scores", "City scores",
    "Formulas. Overall = weighted sum of the three pillar scores (weights on Settings). Rank is within the city; "
    "equal scores share a rank and the next rank follows on (1, 1, 2), as in the I Care workbook.",
    [("City", 20), ("Action ID", 12), ("Action", 60), ("Impact", 10), ("Feasibility", 12), ("Alignment", 11),
     ("Overall score", 13), ("Ranked (1 = yes)", 10), ("Overall rank", 10), ("Key", 30)])
ws_imp = sheet("Impact", "Impact: city impact score",
    "Steps 5–10 of the I Care V15 workbook with AdaptaBrasil city risk. Score is normalised within the city "
    "(top action = 1). Blank = no Direct link, so no impact score.",
    [("City", 20), ("Action ID", 12), ("Action", 60), ("Impact score", 12), ("Impact rank", 11),
     ("Sector of best risk", 26), ("Best risk", 40)])
ws_ass = sheet("Impact assessment", "Impact assessment: action × sector links",
    "Source: I Care Brasil, Adaptation Action Impact Assessment ICB × OEF V15 (7 Oct 2026). Direct and Indirect "
    "links only; Out of Scope links omitted. Only Direct links are scored.",
    [("Action ID", 12), ("Action", 45), ("Sector", 22), ("Component", 13), ("Eligibility", 11), ("Effectiveness", 13),
     ("Confidence", 12), ("Maladaptation", 13), ("Timeline", 11), ("Component score", 11), ("Human reviewed", 10),
     ("Eligibility rationale", 50), ("Effectiveness rationale", 50), ("Confidence rationale", 40),
     ("Maladaptation rationale", 50)])
ws_feas = sheet("Feasibility", "Feasibility: cost and legal feasibility",
    "Sources: cost band from the Action Cost Calculation; legal feasibility score (1–5) from the legal feasibility "
    "score profiles (laws on the Legal basis tab). Feasibility = cost score × cost share + legal score rescaled to "
    "0–1 × legal share (Settings).",
    [("Action ID", 12), ("Action", 62), ("Cost band", 11), ("Cost score", 10), ("Legal score (1–5)", 11),
     ("Legal grade", 28), ("Legal score (0–1)", 11), ("Feasibility score", 12)])
ws_law = sheet("Legal basis", "Legal basis: the federal laws behind each legal feasibility score",
    "Source: legal feasibility score profiles. Authority: total or partial. Competence: exclusive, supplementary or "
    "concurrent/private. Note: the source formula scores partial authority as 0 (it looks for 'parcial').",
    [("Action ID", 12), ("Action", 45), ("Law ID", 9), ("Law", 50), ("Code", 16), ("Type", 20), ("Authority", 10),
     ("Competence", 18), ("Link", 50)])
ws_al = sheet("Alignment", "Alignment: national policy alignment",
    "Source: br-climate-policy-documents v2 results (draft; no rows reviewed yet). Score 0–1.",
    [("Action ID", 12), ("Action", 50), ("Grade", 9), ("Alignment score", 11), ("Document support", 11),
     ("Documents Medium or better", 12), ("Best document", 40), ("PDF page", 8), ("Deciding passage", 60),
     ("Explanation", 60), ("Rows reviewed", 9)])
ws_set = wb.create_sheet("Settings")

In [12]:
# Settings: the only cells meant to be edited (blue on yellow)
ws = ws_set
for c, w in zip("ABC", (42, 12, 70)):
    ws.column_dimensions[c].width = w
ws["A1"], ws["A2"] = "Settings", "Edit the yellow cells. Every score and rank in the workbook updates."
ws["A1"].font, ws["A2"].font = Font(name=FONT, bold=True, size=13), GREY
SETTINGS = [   # row, label, value (None = section heading), note
    (3, "Pillar weights", None, None),
    (4, "Impact weight", WEIGHTS["impact"], "Default 55%."),
    (5, "Feasibility weight", WEIGHTS["feasibility"], "Default 22.5% (the remaining 45% split evenly)."),
    (6, "Alignment weight", WEIGHTS["alignment"], "Default 22.5%."),
    (7, "Total", "=SUM(B4:B6)", '=IF(ABS(B7-1)<0.0001,"OK","Weights must add up to 100%")'),
    (9, "Feasibility: cost and legal", None, None),
    (10, "Cost share", FEAS_SPLIT["cost"], "Share of the feasibility score from cost band. Default 50%."),
    (11, "Legal share", FEAS_SPLIT["legal"], "Share from the legal feasibility score (1–5, rescaled to 0–1). Default 50%."),
    (12, "Total", "=SUM(B10:B11)", '=IF(ABS(B12-1)<0.0001,"OK","Shares must add up to 100%")'),
    (14, "Cost score by cost band", None, None),
    (15, "low", COST_SCORE["low"], "Cost band from the Action Cost Calculation."),
    (16, "medium", COST_SCORE["medium"], None),
    (17, "high", COST_SCORE["high"], None),
    (19, "Only rank actions with an impact score", "Yes" if RANK_ONLY_WITH_IMPACT else "No",
     "Yes: actions with no Direct link (impact 0) get no rank, as in the I Care workbook. No: rank every action."),
]
for r, label, val, note in SETTINGS:
    ws.cell(row=r, column=1, value=label).font = BOLD if val is None else PLAIN
    if val is not None:
        c = ws.cell(row=r, column=2, value=val)
        formula = isinstance(val, str) and val.startswith("=")
        c.font = PLAIN if formula else BLUE
        if not formula:
            c.fill = YELLOW
        if r in (4, 5, 6, 7, 10, 11, 12):
            c.number_format = "0.0%"
    if note:
        ws.cell(row=r, column=3, value=note).font = PLAIN
dv = DataValidation(type="list", formula1='"Yes,No"', allow_blank=False)
ws.add_data_validation(dv)
dv.add("B19")

# Feasibility: cost score and feasibility score are formulas on Settings
last = put(ws_feas, feasibility[["action_id", "action_name", "cost_band"]])
for i, r in enumerate(range(4, last + 1)):
    row = {4: f'=IFERROR(INDEX(Settings!$B$15:$B$17,MATCH(C{r},Settings!$A$15:$A$17,0)),"")',
           5: feasibility.legal_score.iloc[i], 6: feasibility.legal_grade.iloc[i],
           7: f"=(E{r}-1)/4", 8: f"=Settings!$B$10*D{r}+Settings!$B$11*G{r}"}
    for c, v in row.items():
        cell = ws_feas.cell(row=r, column=c, value=v)
        cell.font, cell.number_format = PLAIN, "0.0" if c == 5 else "0.00"
ws_feas.auto_filter.ref = f"A3:H{last}"

put(ws_law, laws.assign(action_name=laws.action_id.map(names))[
    ["action_id", "action_name", "norm_id", "norm_name", "norm_code", "norm_type", "authority", "competence", "norm_link"]])
put(ws_al, alignment, wrap=(9, 10))
last_imp = put(ws_imp, by_city(grid, "rank", "action_id")[["city", "action_id", "action_name", "score", "rank", "sector", "risk"]],
               fmt={4: "0.000"})
ia = impact[impact.eligibility != "Out of Scope"].assign(
    action_name=lambda d: d.action_id.map(names), human_reviewed=lambda d: d.human_reviewed.map({True: "Yes", False: ""}))
put(ws_ass, ia[["action_id", "action_name", "sector", "component", "eligibility", "effectiveness", "confidence",
                "maladaptation", "timeline", "component_score", "human_reviewed", "eligibility_rationale",
                "effectiveness_rationale", "confidence_rationale", "maladaptation_rationale"]],
    wrap=(12, 13, 14, 15), fmt={10: "0.000"})

# City scores: everything but city, action ID and name is a formula
last_cs = put(ws_city, by_city(grid, "action_id")[["city", "action_id", "action_name"]])
rng = lambda c: f"${c}$4:${c}${last_cs}"
for r in range(4, last_cs + 1):
    row = {
        4: f"=SUMIFS(Impact!$D$4:$D${last_imp},Impact!$A$4:$A${last_imp},A{r},Impact!$B$4:$B${last_imp},B{r})",
        5: f"=INDEX(Feasibility!$H$4:$H$41,MATCH(B{r},Feasibility!$A$4:$A$41,0))",
        6: f"=INDEX(Alignment!$D$4:$D$41,MATCH(B{r},Alignment!$A$4:$A$41,0))",
        7: f"=Settings!$B$4*D{r}+Settings!$B$5*E{r}+Settings!$B$6*F{r}",
        8: f'=IF(Settings!$B$19="Yes",IF(D{r}>0,1,0),1)',
        9: f'=IF(H{r}=1,SUMPRODUCT(({rng("A")}=A{r})*({rng("H")}=1)*({rng("G")}>G{r})'
           f'/COUNTIFS({rng("A")},{rng("A")},{rng("H")},{rng("H")},{rng("G")},{rng("G")}))+1,"")',
        10: f'=A{r}&"|"&B{r}',
    }
    for c, v in row.items():
        cell = ws_city.cell(row=r, column=c, value=v)
        cell.font = GREEN if c in (4, 5, 6) else PLAIN
        if c in (4, 5, 6, 7):
            cell.number_format = "0.000"

# Overall ranking: each action's rank in each city, looked up from City scores
ws = ws_over
ws["A1"] = "Brazil adaptation HIAP: overall ranking"
ws["A2"] = ('="Overall score = impact × "&TEXT(Settings!B4,"0.0%")&" + feasibility × "&TEXT(Settings!B5,"0.0%")'
            '&" + alignment × "&TEXT(Settings!B6,"0.0%")&". Change the weights on the Settings tab."')
ws["A3"] = ("Rank 1 = highest priority in that city. Blank = not ranked (no impact score). Rows are sorted by mean rank "
            "at the default weights; re-sort with the filter after changing weights. Draft: alignment grades are unreviewed.")
ws["A1"].font, ws["A2"].font, ws["A3"].font = Font(name=FONT, bold=True, size=14), GREY, GREY
H, FIRST = 5, 6                                  # header row, first city column
END = FIRST + len(city_order)                    # best-rank column
c0, c1 = col(FIRST), col(END - 1)
headers = [("Action ID", 12), ("Action", 62), ("Cost band", 10), ("Legal grade", 16), ("Alignment grade", 11)] \
    + [(c, 13) for c in city_order] + [("Best rank", 10), ("Mean rank", 10)]
for i, (h, w) in enumerate(headers, start=1):
    c = ws.cell(row=H, column=i, value=h)
    c.font, c.fill = BOLD, HEAD
    c.alignment = Alignment(wrap_text=True, vertical="bottom", horizontal="center" if i > 2 else "left")
    ws.column_dimensions[col(i)].width = w
lookup = lambda tab, c, r: f"=INDEX({tab}!${c}$4:${c}$41,MATCH(A{r},{tab}!$A$4:$A$41,0))"
for r, aid in enumerate(order, start=H + 1):
    row = {1: aid, 2: names[aid], 3: lookup("Feasibility", "C", r), 4: lookup("Feasibility", "F", r),
           5: lookup("Alignment", "C", r),
           END: f'=IF(COUNT({c0}{r}:{c1}{r})=0,"",MIN({c0}{r}:{c1}{r}))',
           END + 1: f'=IF(COUNT({c0}{r}:{c1}{r})=0,"",AVERAGE({c0}{r}:{c1}{r}))'}
    for j in range(FIRST, END):
        row[j] = (f"=IFERROR(INDEX('City scores'!$I$4:$I${last_cs},"
                  f"MATCH({col(j)}${H}&\"|\"&$A{r},'City scores'!$J$4:$J${last_cs},0)),\"\")")
    for c, v in row.items():
        cell = ws.cell(row=r, column=c, value=v)
        cell.font = GREEN if 3 <= c < END else PLAIN
        if c > 2:
            cell.alignment = Alignment(horizontal="center")
    ws.cell(row=r, column=END + 1).number_format = "0.0"
last = H + len(order)
ws.conditional_formatting.add(f"{c0}{H + 1}:{c1}{last}", ColorScaleRule(
    start_type="num", start_value=1, start_color="63BE7B", mid_type="percentile", mid_value=50,
    mid_color="FFEB84", end_type="max", end_color="F8F8F8"))
ws.freeze_panes = ws.cell(row=H + 1, column=3)
ws.auto_filter.ref = f"A{H}:{col(END + 1)}{last}"

wb.calculation.fullCalcOnLoad = True
wb.save(WORKBOOK)
print(f"wrote {WORKBOOK.name}")

wrote br_adaptation_hiap_ranking.xlsx


## 4. Comparison with the earlier model

The earlier HIAP model ranked the original adaptation actions for every municipality; its export (`sample/hiap_actions_extract_2026-10-12.csv`) keeps each city's top 20. Action IDs alone can't link the two lists: original actions were merged, split, or kept their ID but changed scope. `data/mapping/previous_to_updated_actions.csv` maps each of the 53 original adaptation actions (ID, name, description) to the updated list:

| Relation | Meaning |
| --- | --- |
| same | Same ID, same intent (wording may be tightened) |
| revised | Same ID, scope narrowed, widened or shifted |
| replaced | Original folded into an updated action with a different ID |
| split | Part of the original moved to another updated action (extra row) |
| no successor | No updated action covers it |
| new | Updated action with no original counterpart |

Each city's old top 20 is translated through the mapping (where several old actions point to one updated action, the best rank is kept) and compared with the new top 20, overall and impact only. If impact alone already differs a lot, the change comes from the impact method, not the weights.

In [13]:
TOP = 20
mapping = pd.read_csv(MAPPING / "previous_to_updated_actions.csv", keep_default_na=False)
LOCODES = grid.drop_duplicates("locode").set_index("locode").city

old = pd.read_csv(OLD_MODEL, low_memory=False)
old = old[(old.type == "adaptation") & old.locode.isin(LOCODES.index)]
old = old.groupby(["locode", "action_id"], as_index=False)["rank"].min()   # Rio de Janeiro has two rankings
assert set(old.action_id) <= set(mapping.previous_action_id), "old action missing from the mapping"

links = mapping[mapping.updated_action_id != ""]
old_new = (old.merge(links, left_on="action_id", right_on="previous_action_id").sort_values("rank")
           .groupby(["locode", "updated_action_id"], as_index=False).first()
           .rename(columns={"rank": "old_rank", "action_id": "via_old_action", "updated_action_id": "action_id"})
           [["locode", "action_id", "old_rank", "via_old_action"]])
lost = old[old.action_id.isin(mapping.loc[mapping.relation == "no successor", "previous_action_id"])]

# One relation per updated action: its own-ID link (same / revised) if any, otherwise replaced or new
own = mapping[mapping.previous_action_id == mapping.updated_action_id].set_index("updated_action_id").relation
other = links[links.previous_action_id != links.updated_action_id]
relation = own.combine_first(other.relation.replace({"split": "replaced"}).groupby(other.updated_action_id).first())
assert set(relation.index) == ACTIONS

both = (grid[["locode", "city", "action_id", "action_name", "overall_rank", "rank"]]
        .rename(columns={"rank": "impact_rank"})
        .merge(old_new, on=["locode", "action_id"], how="left"))
both["relation"] = both.action_id.map(relation)

def status(row, rank_col):
    in_old, in_new = row.old_rank <= TOP, row[rank_col] <= TOP
    if in_old and in_new:
        return "kept"
    if in_new:
        return "entered: new action" if row.relation == "new" else "entered"
    if in_old:
        return "left: not ranked (no impact score)" if pd.isna(row[rank_col]) else "left"
    return ""

for p in ["overall", "impact"]:
    both[f"{p}_status"] = both.apply(status, axis=1, rank_col=f"{p}_rank")
    both[f"{p}_shift"] = both.old_rank - both[f"{p}_rank"]   # positive = higher priority now

both.sort_values(["city", "overall_rank"])[
    ["city", "locode", "action_id", "action_name", "relation", "via_old_action", "old_rank", "overall_rank",
     "overall_status", "overall_shift", "impact_rank", "impact_status", "impact_shift"]
].to_csv(OUTPUT / "comparison_earlier_model.csv", index=False)
print(f"{len(old)} old top-{TOP} entries; {len(lost)} point to an action with no successor")

100 old top-20 entries; 1 point to an action with no successor


**How much of the top of the list is shared?** Of the actions in each city's new top 10 (top 20), how many were also in the old top 10 (top 20). Ties at the cut-off can make a list slightly longer.

In [14]:
def top(g, rank_col, k):
    return set(g.loc[g[rank_col] <= k, "action_id"])

rows = []
for locode, g in both.groupby("locode", sort=False):
    r = {"city": LOCODES[locode]}
    for p, label in [("overall", "overall"), ("impact", "impact only")]:
        for k in (10, 20):
            o, n = top(g, "old_rank", k), top(g, f"{p}_rank", k)
            r[f"{label}: top {k} shared"] = f"{len(o & n)} of {len(n)}"
        r[f"{label}: median shift of kept"] = g.loc[g[f"{p}_status"] == "kept", f"{p}_shift"].median()
    rows.append(r)
pd.DataFrame(rows).set_index("city")

,overall: top 10 shared,overall: top 20 shared,overall: median shift of kept,impact only: top 10 shared,impact only: top 20 shared,impact only: median shift of kept
city,,,,,,
Caxias do Sul,4 of 10,9 of 20,-1.0,4 of 10,10 of 23,-2.5
Sobral,3 of 10,12 of 20,0.0,2 of 10,12 of 21,-1.0
Santa Cruz do Arari,2 of 10,10 of 20,-4.0,2 of 14,12 of 25,-4.5
São Luís,3 of 10,12 of 20,0.5,3 of 10,12 of 22,0.0
Itaubal,2 of 10,9 of 20,-4.0,4 of 15,12 of 25,-2.0


**What moved, per city:** actions that entered or left the overall top 20, and kept actions that moved 5 places or more.

In [15]:
for locode, g in both.groupby("locode", sort=False):
    print(f"\n=== {LOCODES[locode]} ===")
    for label, sel in [("Entered top 20", g.overall_status.str.startswith("entered")),
                       ("Left top 20", g.overall_status.str.startswith("left")),
                       ("Kept, moved 5+ places", (g.overall_status == "kept") & (g.overall_shift.abs() >= 5))]:
        part = g[sel].sort_values("overall_rank")
        if len(part):
            print(f"{label}:")
        for r in part.itertuples():
            o = "-" if pd.isna(r.old_rank) else int(r.old_rank)
            n = "-" if pd.isna(r.overall_rank) else int(r.overall_rank)
            print(f"  {o:>2} -> {n:>2}  {r.action_name[:70]}  [{r.relation}; {r.overall_status}]")
    for r in lost[lost.locode == locode].itertuples():
        print(f"  old rank {r.rank}: {r.action_id} has no successor")


=== Caxias do Sul ===
Entered top 20:
   - ->  1  Improve climate-resilient water and sanitation infrastructure  [revised; entered]
   - ->  2  Strengthen selective waste collection to reduce health risks  [new; entered: new action]
   - ->  5  Secure land tenure, essential services and resilient livelihoods for v  [new; entered: new action]
   - ->  6  Implement and strengthen multi-hazard early warning systems  [same; entered]
   - -> 12  Build and maintain shelters for critical climate hazards  [same; entered]
   - -> 15  Implement integrated coastal zone management measures  [same; entered]
   - -> 16  Reduce pressures on ecosystems to build resilience  [same; entered]
   - -> 17  Strengthen cooperative, intersectoral and participatory climate govern  [revised; entered]
   - -> 18  Stabilize, inspect and maintain slopes in landslide-prone areas  [same; entered]
   - -> 19  Upgrade and maintain urban drainage systems  [revised; entered]
   - -> 20  Build safe housing and implement 